# MIE1624A3 - Indeed Web Scraping Workflow Improvement

## For Assignment 3:
This notebook shows the web scraping workflow I used to build the Assignment 3 dataset. I started from the provided scraper, but my workflow involved more than just changing the search settings.

### Improvements in my workflow
Rather than extending the scraper to additional job websites, I improved the provided Indeed scraping workflow in terms of reliability and usability. Specifically, I adapted it into a multi-search batch workflow, used checkpoint-based reruns for long scraping sessions, merged the batch outputs into a single assignment dataset, and added a second-pass description enrichment step to improve text completeness for downstream NLP analysis.

### Search combinations used
- `data analyst` | `remote` | `US` | `220` jobs
- `data scientist` | `remote` | `US` | `150` jobs
- `data engineer` | `remote` | `US` | `180` jobs
- `machine learning engineer` | `remote` | `US` | `150` jobs
- `data scientist` | `Toronto, ON` | `CA` | `120` jobs
- `machine learning engineer` | `Seattle, WA` | `US` | `120` jobs
- `business analyst` | `remote` | `US` | `137` jobs

### Main scripts used
- `indeed_scraper_camoufox.py` 
    - Main scraper used to collect Indeed postings, extract job metadata, descriptions, salary information when available, and save checkpoint files during long runs. It also reduces repeated postings during scraping by tracking Indeed posting identifiers (job_key, i.e., the Indeed jk extracted from posting links) across pages and resumed runs, and performs a final deduplication pass before saving the batch CSV.
- `merge_indeed_results.py` 
    - Post-processing script used to merge the batch CSV outputs into one assignment dataset and apply another deduplication pass based on Indeed job IDs extracted from posting links.   
- `enrich_indeed_descriptions.py`
    - Enrichment script used after the main scrape to revisit postings with missing descriptions and backfill fuller job text for downstream NLP-based skill extraction. When salary is missing, it can also update the salary field if a usable value is found during the revisit.

### Final file for the assignment
- `webscraping_results_assignment3.csv`

##System Check

In [ ]:
import sys
import platform

print("SYSTEM CHECK")
print(f"OS: {platform.system()}")
print(f"Python: {sys.version_info.major}.{sys.version_info.minor}.{sys.version_info.micro}")

if sys.version_info < (3, 7):
    print("\nERROR: Python 3.7+ required")
else:
    print("\nPython version OK")



##Install Dependencies

In [ ]:
print("Installing required libraries...\n")
!pip install --quiet pandas beautifulsoup4 camoufox camoufox-captcha
print("\nInstallation complete!")


##Verify Dependencies

In [ ]:
import importlib

print("DEPENDENCY CHECK")

required = ['pandas', 'bs4', 'camoufox', 'camoufox_captcha']
all_good = True

for lib in required:
    try:
        importlib.import_module(lib)
        print(f"{lib}")
    except ImportError:
        print(f"{lib} - MISSING")
        all_good = False

if all_good:
    print("All dependencies installed!")
else:
    print("Some dependencies missing - re-run Cell 2")


## Configure a Single Pilot Test Search

- `POSITION`: job title, for example `data analyst`
- `LOCATION`: location, for example `remote` or `Toronto, ON`
- `COUNTRY`: which Indeed site to use for this assignment (`us` or `ca`)
- `MAX_JOBS`: target number of jobs for the test run

Rough time estimates:
- 50 jobs: about 10-15 minutes
- 100 jobs: about 20-30 minutes
- 500 jobs: about 1.5-2.5 hours
- 1000 jobs: about 3-5 hours


In [ ]:
# ===== CONFIGURE SEARCH =====

POSITION = "data analyst"
LOCATION = "remote"
COUNTRY = "ca"
MAX_JOBS = 500

# =================================

print("=" * 70)
print("SEARCH CONFIGURATION")
print("=" * 70)
print(f"Position: {POSITION}")
print(f"Location: {LOCATION}")
print(f"Country:  {COUNTRY}")
print(f"Max Jobs: {MAX_JOBS}")

# Time estimates
est_minutes_low = MAX_JOBS * 0.2
est_minutes_high = MAX_JOBS * 0.3
print(f"\nEstimated time: {est_minutes_low:.0f}-{est_minutes_high:.0f} minutes")
print("=" * 70)


SEARCH CONFIGURATION
Position: data analyst
Location: remote
Country:  ca
Max Jobs: 500

Estimated time: 100-150 minutes


## Run the Main Scraper

This is the main scraping step. This cell also makes the workflow easier to monitor by checking that the script exists, printing run status, and showing whether the scraper finished successfully or should be resumed from a checkpoint.



In [ ]:
import subprocess
import sys
import os
from datetime import datetime

# Locate script
SCRIPT_NAME = "indeed_scraper_camoufox.py"
SCRIPT_PATH = os.path.join(os.getcwd(), SCRIPT_NAME)

if not os.path.exists(SCRIPT_PATH):
    print("="*70)
    print("[X] ERROR: Script not found!")
    print("="*70)
    print(f"\nLooking for: {SCRIPT_NAME}")
    print(f"In directory: {os.getcwd()}")
    print("\n[>] Download the script and place it in the same folder as this notebook.")
    print("\nFiles in current directory:")
    for f in os.listdir('.'):
        if f.endswith('.py'):
            print(f"  - {f}")
else:
    print("="*70)
    print("STARTING SCRAPER")
    print("="*70)
    print(f"Search: {POSITION} in {LOCATION}")
    print(f"Target: {MAX_JOBS} jobs")
    print(f"Start: {datetime.now().strftime('%H:%M:%S')}")
    print("\n" + "="*70)
    print("SCRAPER OUTPUT")
    print("="*70 + "\n")

    try:
        result = subprocess.run(
            [sys.executable, SCRIPT_PATH, POSITION, LOCATION, COUNTRY, str(MAX_JOBS)],
            capture_output=True,
            text=True
        )

        print(result.stdout)

        if result.stderr:
            # Filter out harmless warnings
            stderr_lines = result.stderr.split('\n')
            errors = [line for line in stderr_lines
                     if line and 'charmap' not in line and 'Downloading GeoIP' not in line]
            if errors:
                print("\n" + "="*70)
                print("WARNINGS/ERRORS")
                print("="*70)
                for error in errors:
                    print(error)

        print("\n" + "="*70)
        print(f"End: {datetime.now().strftime('%H:%M:%S')}")
        print("="*70)

        if result.returncode == 0:
            print("\n[OK] SCRAPING COMPLETED!")
            print("\n[>] Proceed to Cell 6 to analyze results")
        else:
            print(f"\n[!] Scraper exited with code: {result.returncode}")
            print("\n[>] Try running this cell again - it will resume from checkpoint")

    except Exception as e:
        print("\n" + "="*70)
        print("[X] ERROR")
        print("="*70)
        print(f"\n{e}")
        print("\n[HELP] Check troubleshooting section below")

STARTING SCRAPER
Search: data analyst in remote
Target: 500 jobs
Start: 10:52:40

SCRAPER OUTPUT



## Load Workflow Output

This cell is to load the most useful file from the scraping workflow instead of just picking an arbitrary recent CSV. It first looks for the enriched final file, then the merged assignment file, and only falls back to the most recent CSV if neither is available.


In [ ]:
import pandas as pd
import glob
import os
from pathlib import Path

print('LOADING WORKFLOW OUTPUT')

preferred_files = [
    'webscraping_results_assignment3_enriched.csv',
    'webscraping_results_assignment3.csv',
]

selected_file = None
for filename in preferred_files:
    if Path(filename).exists():
        selected_file = filename
        break

if selected_file is None:
    csv_files = [f for f in glob.glob('*.csv') if not f.startswith('checkpoint')]
    if csv_files:
        selected_file = max(csv_files, key=os.path.getctime)

if selected_file:
    print()
    print('Loading:', selected_file)
    df = pd.read_csv(selected_file)

    print()
    print('DATA SUMMARY')
    print('Total jobs:', len(df))

    if 'Description' in df.columns:
        nonempty_desc = df['Description'].fillna('').astype(str).str.strip().ne('').sum()
        print('Jobs with description:', nonempty_desc, f'({nonempty_desc/len(df)*100:.1f}%)')

    if 'Salary' in df.columns:
        jobs_with_salary = (df['Salary'].fillna('N/A') != 'N/A').sum()
        print('Jobs with salary:', jobs_with_salary, f'({jobs_with_salary/len(df)*100:.1f}%)')

    if 'Company' in df.columns:
        print('Unique companies:', df['Company'].nunique())
    if 'Location' in df.columns:
        print('Unique locations:', df['Location'].nunique())

    print()
    print('PREVIEW (First 5 rows)')
    display(df.head())

    print()
    print('Data loaded successfully')
else:
    print()
    print('No results found')
    print()
    print('Possible reasons:')
    df = None



##Quick Summary of the Current Output

This is a basic EDA of the currently loaded scraping output. The goal here is just to check dataset size, missingness, text coverage, salary coverage, and a few quick examples from the main columns before moving on.

This also acts as a quick quality check before moving on to batch runs, merging, and enrichment.


In [ ]:
if df is not None:
    print("BASIC EDA SUMMARY")
    print()
    print(f"Rows: {len(df)}")
    print(f"Columns: {df.shape[1]}")
    print()
    print("Missing values by column:")
    missing_counts = df.isna().sum().sort_values(ascending=False)
    display(missing_counts.to_frame(name="missing_count"))

    if "Description" in df.columns:
        desc_nonempty = df["Description"].fillna("").astype(str).str.strip().ne("").sum()
        desc_missing = len(df) - desc_nonempty
        print(f"Description present: {desc_nonempty} ({desc_nonempty/len(df)*100:.2f}%)")
        print(f"Description missing: {desc_missing} ({desc_missing/len(df)*100:.2f}%)")

    if "Salary" in df.columns:
        salary_missing = df["Salary"].isna() | df["Salary"].astype(str).str.strip().isin(["", "N/A"])
        salary_present = (~salary_missing).sum()
        print(f"Salary present: {salary_present} ({salary_present/len(df)*100:.2f}%)")
        print(f"Salary missing: {salary_missing.sum()} ({salary_missing.sum()/len(df)*100:.2f}%)")

    if "Company" in df.columns:
        print(f"Unique companies: {df['Company'].nunique(dropna=True)}")
    if "Location" in df.columns:
        print(f"Unique locations: {df['Location'].nunique(dropna=True)}")
    if "Links" in df.columns:
        print(f"Unique links: {df['Links'].nunique(dropna=True)}")

    print()
    print("Example non-null values from key columns:")
    preview_cols = [c for c in ["Title", "Company", "Location", "Salary", "Description"] if c in df.columns]
    for col in preview_cols:
        sample_vals = df[col].dropna().astype(str)
        sample_vals = sample_vals[sample_vals.str.strip().ne("")].head(3).tolist()
        print(f"- {col}: {sample_vals}")
else:
    print("No data to analyze - run Cell 6 first")


---

## Run the Full Multi-Search Workflow

This cell runs the full set of searches that I used to build the assignment dataset.

Instead of relying on a single search, I ran several searches across analyst, scientist, engineer, and business-focused roles, as well as different locations. This made the final dataset broader and more balanced, which is helpful later because the NLP and clustering steps depend on the wording used in the job postings.

This step also turns the scraper into a batch workflow rather than a single standalone run. Each search is run one after another, the outputs are saved as separate batch CSV files, and those files are used later in the merge step. The workflow also pauses briefly between runs and keeps the outputs organized for later processing.

Since the scraper supports checkpoints, interrupted runs can usually be resumed with the same search settings instead of starting over.

##Improvement in this step
1. I used a multi-search workflow instead of relying on only one query.
2. I collected postings across a wider mix of roles and locations.
3. I organized the scraping process as batch runs that could be merged later into one assignment dataset.


In [ ]:
import subprocess
import sys
import time
import glob
import os

# ============================================================
# MULTI-SEARCH CONFIGURATION USED FOR THE ASSIGNMENT
# ============================================================
searches = [
    {'position': 'data analyst',              'location': 'remote',       'country': 'us', 'max_jobs': 220},
    {'position': 'data scientist',            'location': 'remote',       'country': 'us', 'max_jobs': 150},
    {'position': 'data engineer',             'location': 'remote',       'country': 'us', 'max_jobs': 180},
    {'position': 'machine learning engineer', 'location': 'remote',       'country': 'us', 'max_jobs': 150},
    {'position': 'data scientist',            'location': 'Toronto, ON',  'country': 'ca', 'max_jobs': 120},
    {'position': 'machine learning engineer', 'location': 'Seattle, WA',  'country': 'us', 'max_jobs': 120},
    {'position': 'business analyst',          'location': 'remote',       'country': 'us', 'max_jobs': 137},
]

print(f'RUNNING {len(searches)} SEARCHES')
total_jobs = sum(s['max_jobs'] for s in searches)
est_hours = total_jobs * 0.25 / 60
print()
print(f'Estimated total time: {est_hours:.1f} hours')
print(f'Total target jobs: {total_jobs}')
print('This matches the set of searches I used to build the final dataset.')

for i, search in enumerate(searches, 1):
    print()
    print(f'SEARCH {i}/{len(searches)}')
    print(f"Position: {search['position']}")
    print(f"Location: {search['location']}")
    print(f"Country:  {search['country'].upper()}")
    print(f"Max Jobs: {search['max_jobs']}")
    print()

    result = subprocess.run(
        [sys.executable, SCRIPT_PATH,
         search['position'],
         search['location'],
         search['country'],
         str(search['max_jobs'])],
        capture_output=True,
        text=True
    )

    print(result.stdout)

    if result.stderr:
        stderr_lines = result.stderr.split('\n')
        errors = [line for line in stderr_lines if line and 'charmap' not in line and 'Downloading GeoIP' not in line]
        if errors:
            print('\nWarnings:')
            for error in errors[:5]:
                print(f'  {error}')

    print()
    if result.returncode == 0:
        print(f'Search {i} completed!')
        for line in result.stdout.split('\n'):
            if 'Final results saved to:' in line:
                print(f'   {line.strip()}')
                break
    else:
        print(f'Search {i} failed (exit code: {result.returncode})')
        print('    You can re-run just this search to resume')

    if i < len(searches):
        wait_time = 60
        print()
        print(f'Waiting {wait_time} seconds before next search...')
        time.sleep(wait_time)

print()
print('ALL SEARCHES COMPLETE')
print()
print('Batch CSV files currently available:')
csv_files = [f for f in glob.glob('*.csv') if not f.startswith('checkpoint')]
csv_files.sort(key=os.path.getctime, reverse=True)
for f in csv_files:
    size = os.path.getsize(f) / 1024
    print(f'   - {f} ({size:.1f} KB)')

print(f'\nTotal non-checkpoint CSV files: {len(csv_files)}')



---

## Merge the Batch CSV Files

After running multiple searches, I used a merge step to combine the individual batch CSV files into one assignment dataset.

Without merging, the results would still be split across separate search files.


In [ ]:
import subprocess
import sys

MERGE_SCRIPT = "merge_indeed_results.py"

if not os.path.exists(MERGE_SCRIPT):
    print(f"{MERGE_SCRIPT} not found in the current folder")
else:
    print("MERGING BATCH CSV FILES")
    merge_result = subprocess.run(
        [sys.executable, MERGE_SCRIPT],
        capture_output=True,
        text=True
    )
    print(merge_result.stdout)
    if merge_result.stderr:
        print(merge_result.stderr)
    print(f"Merge return code: {merge_result.returncode}")



## Enrich Missing Descriptions

After the initial scrape, many rows still had missing descriptions, which is a problem for the later NLP and clustering parts of the assignment. So I ran a second pass that revisited the Indeed links and tried to backfill the missing job text.

This step can take a while because it opens the browser again and visits job links one by one.

This improves field completeness for the text data, which is important because the later NLP pipeline depends on description coverage.


In [ ]:
import subprocess
import sys

ENRICH_SCRIPT = "enrich_indeed_descriptions.py"
INPUT_CSV = "webscraping_results_assignment3.csv"
OUTPUT_CSV = "webscraping_results_assignment3_enriched.csv"
RUN_ENRICH = False  # change to True only when you are ready to run the enrichment step

if not RUN_ENRICH:
    print("RUN_ENRICH is False. Set it to True for backfill missing descriptions.")
elif not os.path.exists(ENRICH_SCRIPT):
    print(f"{ENRICH_SCRIPT} not found in the current folder")
elif not os.path.exists(INPUT_CSV):
    print(f"{INPUT_CSV} not found. ")
else:
    print("ENRICHING MISSING DESCRIPTIONS")
    enrich_result = subprocess.run(
        [sys.executable, ENRICH_SCRIPT, "--input", INPUT_CSV, "--output", OUTPUT_CSV],
        capture_output=True,
        text=True
    )
    print(enrich_result.stdout)
    if enrich_result.stderr:
        print(enrich_result.stderr)
    print(f"Enrich return code: {enrich_result.returncode}")



## Check Merge and Enrichment Results

This final summary makes it easy to see how the workflow changed the data.

What I mainly wanted to check here was:
- how many rows were in the merged dataset
- how many descriptions were available before enrichment
- how much the description coverage improved after enrichment


In [ ]:
import pandas as pd
from pathlib import Path

merged_file = Path("webscraping_results_assignment3.csv")
enriched_file = Path("webscraping_results_assignment3_enriched.csv")

def description_stats(df, label):
    if "Description" not in df.columns:
        print(f"[{label}] No Description column found")
        return
    nonempty = df["Description"].fillna("").astype(str).str.strip().ne("").sum()
    missing = len(df) - nonempty
    print(f"[{label}] rows: {len(df)}")
    print(f"[{label}] descriptions present: {nonempty} ({nonempty/len(df)*100:.2f}%)")
    print(f"[{label}] descriptions missing: {missing} ({missing/len(df)*100:.2f}%)")

print("POST-PROCESSING SUMMARY")

if merged_file.exists():
    merged_df = pd.read_csv(merged_file)
    description_stats(merged_df, "Merged CSV")
else:
    print("Merged CSV not found yet.")

print("-" * 70)

if enriched_file.exists():
    enriched_df = pd.read_csv(enriched_file)
    description_stats(enriched_df, "Enriched CSV")
else:
    print("Enriched CSV not found yet.")
